# Feature Engineering (Implicit Feedback for NCF)



## Step 1: Load the cleaned data

In [1]:
import pandas as pd
import numpy as np
import os

PROCESSED_DIR = "data/processed"
FEATURE_DIR = "data/features"
os.makedirs(FEATURE_DIR, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df_business = pd.read_csv(f"{PROCESSED_DIR}/business_clean.csv")
df_review = pd.read_csv(f"{PROCESSED_DIR}/review_clean.csv")

print("Businesses:", df_business.shape)
print("Reviews:", df_review.shape)

Businesses: (150243, 59)
Reviews: (138747, 10)


## Step 2: Keep Only Restaurants


In [2]:
df_business["categories"] = df_business["categories"].fillna("")
df_restaurants = df_business[df_business["categories"].str.contains("Restaurants", case=False)]
df_restaurants = df_restaurants.reset_index(drop=True)

print("Restaurants kept:", df_restaurants.shape[0], "out of", df_business.shape[0], "total businesses")

Restaurants kept: 52268 out of 150243 total businesses


## Step 3 (Optional): Keep Only Top Cities


In [3]:
city_counts = df_restaurants["city"].value_counts()
print(city_counts.head(10))

TOP_N_CITIES = 30
top_cities = city_counts.head(TOP_N_CITIES).index.tolist()
print("\nKeeping these cities:", top_cities)

df_restaurants = df_restaurants[df_restaurants["city"].isin(top_cities)]
df_restaurants = df_restaurants.reset_index(drop=True)

print("Restaurants after city filter:", df_restaurants.shape[0])

city
Philadelphia    5852
Tampa           2960
Indianapolis    2862
Nashville       2502
Tucson          2466
New Orleans     2259
Edmonton        2166
Saint Louis     1790
Reno            1286
Boise            847
Name: count, dtype: int64

Keeping these cities: ['Philadelphia', 'Tampa', 'Indianapolis', 'Nashville', 'Tucson', 'New Orleans', 'Edmonton', 'Saint Louis', 'Reno', 'Boise', 'Santa Barbara', 'Clearwater', 'Wilmington', 'St. Louis', 'Metairie', 'Saint Petersburg', 'Franklin', 'St. Petersburg', 'Sparks', 'Brandon', 'Meridian', 'Largo', 'Cherry Hill', 'Carmel', 'West Chester', 'Kenner', 'New Port Richey', 'Goleta', 'Palm Harbor', 'Greenwood']
Restaurants after city filter: 32696


## Step 4: Turn Reviews AND Tips into Labeled Interactions

In [4]:
restaurant_ids = set(df_restaurants["business_id"])

POSITIVE_THRESHOLD = 3

df_tip = pd.read_csv(f"{PROCESSED_DIR}/tip_clean.csv")

# Reviews -> label based on stars
df_review_restaurants = df_review[df_review["business_id"].isin(restaurant_ids)].copy()
df_review_restaurants["label"] = (df_review_restaurants["stars"] >= POSITIVE_THRESHOLD).astype(int)
df_review_restaurants = df_review_restaurants[["user_id", "business_id", "date", "label"]]

# Tips -> always treated as a positive interaction (engagement, no star rating available)
df_tip_restaurants = df_tip[df_tip["business_id"].isin(restaurant_ids)][["user_id", "business_id", "date"]].copy()
df_tip_restaurants["label"] = 1

# Combine reviews + tips into one interactions table
df_interactions = pd.concat([df_review_restaurants, df_tip_restaurants], ignore_index=True)
df_interactions["date"] = pd.to_datetime(df_interactions["date"])
df_interactions = df_interactions.sort_values("date")
df_interactions = df_interactions.drop_duplicates(subset=["user_id", "business_id"], keep="last")

print("Total labeled interactions (reviews + tips):", df_interactions.shape[0])
print(df_interactions["label"].value_counts())

Total labeled interactions (reviews + tips): 72535
label
1    60480
0    12055
Name: count, dtype: int64


## Step 4b: Filter to Users & Restaurants With Enough Interactions (k-core filtering)


In [5]:
def k_core_filter(df, min_interactions=3):
    df = df.copy()
    while True:
        user_counts = df.groupby("user_id").size()
        item_counts = df.groupby("business_id").size()

        valid_users = user_counts[user_counts >= min_interactions].index
        valid_items = item_counts[item_counts >= min_interactions].index

        filtered = df[df["user_id"].isin(valid_users) & df["business_id"].isin(valid_items)]

        if len(filtered) == len(df):   # nothing more removed, stable
            break
        df = filtered

    return df.reset_index(drop=True)

MIN_INTERACTIONS = 3

df_interactions = k_core_filter(df_interactions, min_interactions=MIN_INTERACTIONS)

print("Interactions after k-core filtering:", df_interactions.shape[0])
print("Unique users:", df_interactions['user_id'].nunique())
print("Unique restaurants:", df_interactions['business_id'].nunique())

# Sanity check: confirm sparsity is actually fixed
check_counts = df_interactions.groupby("user_id").size()
print("\nUsers with only 1 interaction (should be 0):", (check_counts == 1).sum())
print("Users with 5+ interactions:", (check_counts >= 5).sum())

Interactions after k-core filtering: 7001
Unique users: 1815
Unique restaurants: 440

Users with only 1 interaction (should be 0): 0
Users with 5+ interactions: 345


## Step 5: Convert User & Restaurant IDs into Simple Numbers


In [6]:
unique_user_ids = df_interactions["user_id"].unique()
unique_item_ids = df_interactions["business_id"].unique()

user_id_map = {}
for i, uid in enumerate(unique_user_ids):
    user_id_map[uid] = i

item_id_map = {}
for i, bid in enumerate(unique_item_ids):
    item_id_map[bid] = i

df_interactions["user_idx"] = df_interactions["user_id"].map(user_id_map)
df_interactions["item_idx"] = df_interactions["business_id"].map(item_id_map)

N_USERS = len(user_id_map)
N_ITEMS = len(item_id_map)

print("Number of unique users:", N_USERS)
print("Number of unique restaurants:", N_ITEMS)

# Save these mappings so we can use them later during model inference
pd.Series(user_id_map).to_csv(f"{FEATURE_DIR}/user_id_map.csv", header=["user_idx"])
pd.Series(item_id_map).to_csv(f"{FEATURE_DIR}/item_id_map.csv", header=["item_idx"])

Number of unique users: 1815
Number of unique restaurants: 440


## Step 6: Record Which Restaurants Each User Has Already Reviewed


In [7]:
user_seen_items = df_interactions.groupby("user_idx")["item_idx"].apply(set).to_dict()

## Step 7: Leave-One-Out Split (Train / Test)


In [8]:
df_positive = df_interactions[df_interactions["label"] == 1]
df_negative = df_interactions[df_interactions["label"] == 0]

# Only users with at least 2 liked restaurants can have both a train and a test example
positive_counts = df_positive.groupby("user_idx").size()
eligible_users = positive_counts[positive_counts >= 2].index

df_positive_eligible = df_positive[df_positive["user_idx"].isin(eligible_users)]
df_positive_eligible = df_positive_eligible.sort_values(["user_idx", "date"])

# The last (most recent) positive per user goes to test
test_positive = df_positive_eligible.groupby("user_idx").tail(1)
train_positive = df_positive_eligible.drop(test_positive.index)

# Real negative reviews (label = 0) for eligible users go into training only
train_negative_explicit = df_negative[df_negative["user_id"].isin(
    df_positive_eligible["user_id"]
)]

print("Train positive examples:", train_positive.shape[0])
print("Train explicit negative examples (real bad reviews):", train_negative_explicit.shape[0])
print("Test positive examples (1 per eligible user):", test_positive.shape[0])

Train positive examples: 4499
Train explicit negative examples (real bad reviews): 549
Test positive examples (1 per eligible user): 1751


## Step 8: Add Random Negative Examples


In [9]:
def sample_random_negatives(n, items_to_avoid):
    """Randomly pick n restaurant indices that are NOT in items_to_avoid."""
    negatives = []
    while len(negatives) < n:
        candidate = np.random.randint(0, N_ITEMS)
        if candidate not in items_to_avoid:
            negatives.append(candidate)
    return negatives

N_NEG_RANDOM_TRAIN = 2   # random negatives per train positive
N_NEG_TEST = 99          # random negatives per test user (standard: 1 positive + 99 negatives)

In [10]:
# Builing random negative for training
train_random_neg_rows = []

for _, row in train_positive.iterrows():
    user = row["user_idx"]
    negatives = sample_random_negatives(N_NEG_RANDOM_TRAIN, user_seen_items[user])
    for item in negatives:
        train_random_neg_rows.append({"user_idx": user, "item_idx": item, "label": 0})

df_train_random_neg = pd.DataFrame(train_random_neg_rows)
print("Random train negatives created:", df_train_random_neg.shape[0])

Random train negatives created: 8998


In [11]:
# Building random negative for testing
test_random_neg_rows = []

for _, row in test_positive.iterrows():
    user = row["user_idx"]
    negatives = sample_random_negatives(N_NEG_TEST, user_seen_items[user])
    for item in negatives:
        test_random_neg_rows.append({"user_idx": user, "item_idx": item, "label": 0})

df_test_random_neg = pd.DataFrame(test_random_neg_rows)
print("Random test negatives created:", df_test_random_neg.shape[0])

Random test negatives created: 173349


## Step 9: Combine Everything into Final Train & Test Sets

In [12]:
df_train_final = pd.concat([
    train_positive[["user_idx", "item_idx", "label"]],
    train_negative_explicit[["user_idx", "item_idx", "label"]],
    df_train_random_neg
], ignore_index=True)

# Shuffle so positives and negatives are mixed, not grouped together
df_train_final = df_train_final.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

df_test_final = pd.concat([
    test_positive[["user_idx", "item_idx", "label"]],
    df_test_random_neg
], ignore_index=True)

print("Final train set:", df_train_final.shape[0], "rows")
print(df_train_final["label"].value_counts())

print("\nFinal test set:", df_test_final.shape[0], "rows")
print(df_test_final["label"].value_counts())

Final train set: 14046 rows
label
0    9547
1    4499
Name: count, dtype: int64

Final test set: 175100 rows
label
0    173349
1      1751
Name: count, dtype: int64


## Step 10: Saving Everything

In [13]:
df_train_final.to_csv(f"{FEATURE_DIR}/train_interactions.csv", index=False)
df_test_final.to_csv(f"{FEATURE_DIR}/test_interactions.csv", index=False)

# Also save restaurant metadata (useful later for cold-start / content-based features)
df_restaurants_out = df_restaurants[df_restaurants["business_id"].isin(item_id_map.keys())].copy()
df_restaurants_out["item_idx"] = df_restaurants_out["business_id"].map(item_id_map)
df_restaurants_out.to_csv(f"{FEATURE_DIR}/restaurant_metadata.csv", index=False)

print("Saved files:", os.listdir(FEATURE_DIR))

Saved files: ['item_id_map.csv', 'restaurant_metadata.csv', 'test_interactions.csv', 'user_id_map.csv', 'train_interactions.csv']


## Step 11: Sanity Check

In [14]:
train_check = pd.read_csv(f"{FEATURE_DIR}/train_interactions.csv")
test_check = pd.read_csv(f"{FEATURE_DIR}/test_interactions.csv")

print("Train shape:", train_check.shape)
print("Test shape:", test_check.shape)
print("\nNumber of users:", N_USERS)
print("Number of restaurants:", N_ITEMS)

Train shape: (14046, 3)
Test shape: (175100, 3)

Number of users: 1815
Number of restaurants: 440
